# Baseline: ELO-Only Predictor

Maps ELO difference to Poisson goal rates via the win-probability formula — zero trained parameters.

Uses LOTO-CV: 4 folds × 64 matches = 256 pooled evaluation matches. Reports bootstrap 95% CI.

Score prediction: mode of Poisson = floor(λ). Also compares against the Sporza-optimal strategy
(maximise expected Sporza pts over the joint score distribution).

In [1]:
import sys
sys.path.insert(0, '../../src')

import numpy as np
import pandas as pd
import mlflow
from pathlib import Path

from evaluation.sporza import score_breakdown, sporza_points_series

DATA = Path('../../data/processed')
WC_YEARS = [2010, 2014, 2018, 2022]

In [2]:
def bootstrap_ci(pts: pd.Series, n_boot: int = 10000) -> dict:
    rng = np.random.default_rng(42)
    vals = pts.values
    boot = np.array([rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {'mean': float(vals.mean()), 'ci_lo': float(lo), 'ci_hi': float(hi)}


def elo_to_lambda(elo_diff: pd.Series, base_goals: float) -> tuple[pd.Series, pd.Series]:
    p_home = 1.0 / (1.0 + 10.0 ** (-elo_diff / 400.0))
    lh = p_home * base_goals * 2
    la = (1 - p_home) * base_goals * 2
    return lh, la

In [3]:
all_pts = []
fold_results = []

for year in WC_YEARS:
    train = pd.read_parquet(DATA / f'train_fold_{year}.parquet')
    evalf = pd.read_parquet(DATA / f'eval_fold_{year}.parquet')

    base_goals = (train['home_score'].mean() + train['away_score'].mean()) / 2
    lh, la = elo_to_lambda(evalf['elo_diff'], base_goals)

    pred_home = np.floor(lh).astype(int)
    pred_away = np.floor(la).astype(int)

    bd = score_breakdown(pred_home, pred_away, evalf['home_score'].values, evalf['away_score'].values)
    pts = sporza_points_series(pred_home, pred_away,
                               evalf['home_score'].reset_index(drop=True),
                               evalf['away_score'].reset_index(drop=True))
    all_pts.extend(pts.tolist())
    fold_results.append({'year': year, 'mean_pts': bd['mean_pts'], 'pct_exact': bd['pct_exact'],
                         'pct_correct_result': bd['pct_correct_result'], 'base_goals': base_goals})
    print(f'WC {year}: base_goals={base_goals:.3f}  mean_pts={bd["mean_pts"]:.3f}  '
          f'exact={bd["pct_exact"]:.1%}  correct_result={bd["pct_correct_result"]:.1%}')

pooled_pts = pd.Series(all_pts)
ci = bootstrap_ci(pooled_pts)
bd_pool = score_breakdown(
    pd.Series([r for r in all_pts]),  # dummy — just for breakdown
    pd.Series([r for r in all_pts]),
    pd.Series([r for r in all_pts]),
    pd.Series([r for r in all_pts]),
)
print(f'\nPooled LOTO-CV ({len(pooled_pts)} matches): {ci["mean"]:.3f} pts/match  '
      f'95% CI [{ci["ci_lo"]:.3f}, {ci["ci_hi"]:.3f}]  '
      f'CI width={ci["ci_hi"]-ci["ci_lo"]:.3f}')

WC 2010: base_goals=1.237  mean_pts=3.875  exact=12.5%  correct_result=51.6%
WC 2014: base_goals=1.316  mean_pts=3.812  exact=6.2%  correct_result=54.7%
WC 2018: base_goals=1.320  mean_pts=3.891  exact=14.1%  correct_result=51.6%
WC 2022: base_goals=1.322  mean_pts=4.266  exact=14.1%  correct_result=56.2%

Pooled LOTO-CV (256 matches): 3.961 pts/match  95% CI [3.582, 4.352]  CI width=0.770


In [4]:
# Fold-by-fold breakdown
print('Per-fold summary:')
for r in fold_results:
    print(f'  WC {r["year"]}: {r["mean_pts"]:.3f} pts  exact={r["pct_exact"]:.1%}  correct={r["pct_correct_result"]:.1%}')

Per-fold summary:
  WC 2010: 3.875 pts  exact=12.5%  correct=51.6%
  WC 2014: 3.812 pts  exact=6.2%  correct=54.7%
  WC 2018: 3.891 pts  exact=14.1%  correct=51.6%
  WC 2022: 4.266 pts  exact=14.1%  correct=56.2%


In [5]:
mlflow.set_tracking_uri('sqlite:////Users/seppe.vanswegenoven/projects/wk_2026_match_predictor/mlflow.db')
mlflow.set_experiment('wk2026-tabular-2026-06-13')

with mlflow.start_run(run_name='elo_only_loto'):
    mlflow.set_tags({'modality': 'tabular', 'approach': 'elo_only',
                     'eval_strategy': 'loto_cv', 'dataset_version': 'split_v2'})
    mlflow.log_params({'elo_scale': 400, 'rounding': 'floor'})
    mlflow.log_metric('loto_mean_sporza_pts', ci['mean'])
    mlflow.log_metric('loto_ci_lo', ci['ci_lo'])
    mlflow.log_metric('loto_ci_hi', ci['ci_hi'])
    for r in fold_results:
        mlflow.log_metric(f'fold_{r["year"]}_mean_pts', r['mean_pts'])
    run_id = mlflow.active_run().info.run_id

print(f'run_id: {run_id}')
print(f'\nSummary: {ci["mean"]:.3f} [{ci["ci_lo"]:.3f}, {ci["ci_hi"]:.3f}]')

run_id: 822e144c3efc4e30aec897baac830226

Summary: 3.961 [3.582, 4.352]
